# Viagens para a Índia / Voyages to India

Baseado em / Based on

> Wicki, J. (1967). Liste der Jesuiten-Indienfahrer 1541-1758. Aschendorff. https://books.google.com/books?id=bEoXzgEACAAJ



In [ ]:
from timelink.notebooks import TimelinkNotebook

tlnb = TimelinkNotebook(kleio_image="kleio-server", kleio_version="12.9.592")
tlnb.print_info()

## List all who travelled in the same "armada"

Wicki, J. (1967). Liste der Jesuiten-Indienfahrer 1541-1758. Aschendorff. https://books.google.com/books?id=bEoXzgEACAAJ


In [ ]:
import pyuca  # to sort accented characters properly

import pandas as pd
from timelink.pandas import entities_with_attribute
# show 500 rows
pd.set_option('display.max_rows', 550)

voyage_of_interest = '89'

travellers = entities_with_attribute(
    entity_type='person',
    show_elements=['id','name','groupname'],
    the_type='wicky-viagem',  # fix the attribute in db and then here wicki not wicky
    the_value=voyage_of_interest,
    more_attributes=['embarque','jesuita-entrada','nacionalidade'],
    db=tlnb.db,
)
travellers.info()
# filter groupname = 'n' (avoid "referido", "pai", "mãe")
travellers_all_all = travellers[travellers.groupname=='n']
# only those who embarked in the same date
# travellers = travellers[travellers['embarque.date'] == travellers['wicky-viagem.date']]
travellers['wicky-viagem.date.year'] = travellers['wicky-viagem.date'].str[:4]
travellers['embarque.date.year'] = travellers['embarque.date'].str[:4]
# only same year (some do the voyage more than once, in different years)
travellers = travellers[travellers['wicky-viagem.date.year'] == travellers['embarque.date.year']]
travellers.info()

In [ ]:
cols = ['embarque','name','nacionalidade','jesuita-entrada','wicky-viagem.date.year','wicky-viagem','embarque.date.year']
travellers[cols].sort_values(by=['embarque','name','jesuita-entrada','nacionalidade','wicky-viagem.date.year','embarque.date.year',])

In [ ]:
table = travellers[['embarque','name','embarque.date.year']].to_markdown()
print(table)



### Number of missionaries travelling over time

In [ ]:
travellers_all = entities_with_attribute(
    entity_type='person',
    show_elements=['id','name','groupname'],
    the_type='wicky-viagem',  # fix the attribute in db and then here wicki not wicky
    more_attributes=['embarque','jesuita-entrada','nacionalidade'],
    db=tlnb.db,
)
travellers_all.info()

#### Evolution

In [ ]:
# filter groupname = 'n' (avoid "referido", "pai", "mãe")
travellers_all = travellers_all[travellers_all.groupname=='n']
travellers_all['wicky-viagem.date.year'] = travellers_all['wicky-viagem.date'].str[:4].astype(int)
travellers_all['embarque.date.year'] = travellers_all['embarque.date'].str[:4].astype(int)
travellers_all.info()
# group by wicky-viagem, wicky-viagem.date, count
travellers_all_grouped = travellers_all.groupby(['wicky-viagem.date.year','wicky-viagem',]).size().reset_index(name='count')
travellers_all_grouped.info()
travellers_all_grouped.sort_values(by=['wicky-viagem.date.year',])


### Display graph

In [ ]:
import pandas as pd

# Calculate the moving average
travellers_all_grouped['moving_average'] = travellers_all_grouped['count'].rolling(window=10).mean()

plt.figure(figsize=(20, 10))
sns.lineplot(data=travellers_all_grouped, x='wicky-viagem.date.year', y='count', label='Number of travellers')
sns.lineplot(data=travellers_all_grouped, x='wicky-viagem.date.year', y='moving_average', label='10-Year Moving Average')
plt.title('Number of travellers per voyage')
plt.xlabel('Year')
plt.ylabel('Number of travellers')
plt.xticks(range(1540, 1760, 10))

# label data points with the voyage name
for i, row in travellers_all_grouped.iterrows():
    plt.text(row['wicky-viagem.date.year'], row['count'], row['wicky-viagem'], rotation=45)

plt.legend()
plt.show()

In [ ]:
import datetime

# Start and end dates
start_date = datetime.date(1755, 1, 1)
end_date = datetime.date(1758, 12, 31)

# Calculate the difference
diff = end_date - start_date

# Calculate the middle point
mid_point = diff // 2

# Calculate the middle date
middle_date = start_date + mid_point

print(middle_date)